# Project 70 — Local Agent Trace Analyzer
## Instrument, Log & Analyze Agent Execution Traces

**Stack:** LangChain · Ollama · pandas · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pandas

## Step 1 — Build Trace Logger

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import json, time, pandas as pd
from datetime import datetime
from dataclasses import dataclass, field, asdict

llm = ChatOllama(model="qwen3:8b", temperature=0.1)

@dataclass
class TraceStep:
    name: str
    input_summary: str
    output_summary: str
    duration_s: float
    tokens_est: int
    success: bool
    error: str = ""

@dataclass
class Trace:
    task_id: str
    task: str
    start_time: str = ""
    steps: list = field(default_factory=list)
    status: str = "running"
    total_duration: float = 0.0

class TraceCollector:
    def __init__(self):
        self.traces: list[Trace] = []
        self._current: Trace = None
        self._step_start: float = 0

    def begin(self, task_id, task):
        self._current = Trace(task_id=task_id, task=task,
                              start_time=datetime.now().isoformat())

    def step_start(self, name, input_data):
        self._step_start = time.time()
        self._step_name = name
        self._step_input = str(input_data)[:100]

    def step_end(self, output_data, success=True, error=""):
        duration = time.time() - self._step_start
        self._current.steps.append(TraceStep(
            name=self._step_name,
            input_summary=self._step_input,
            output_summary=str(output_data)[:100],
            duration_s=round(duration, 3),
            tokens_est=len(str(output_data).split()),
            success=success, error=error,
        ))

    def end(self, status="success"):
        self._current.status = status
        self._current.total_duration = sum(s.duration_s for s in self._current.steps)
        self.traces.append(self._current)
        self._current = None

collector = TraceCollector()
print("Trace collector ready")

## Step 2 — Run Traced Agent Tasks

In [ ]:
tasks = [
    ("T1", "Summarize the benefits of containerization"),
    ("T2", "List 5 sorting algorithms with time complexities"),
    ("T3", "Explain the CAP theorem in distributed systems"),
    ("T4", "Compare REST vs GraphQL APIs"),
    ("T5", "What is eventual consistency?"),
]

chain = ChatPromptTemplate.from_template("{task}") | llm | StrOutputParser()

for task_id, task in tasks:
    collector.begin(task_id, task)
    try:
        # Step 1: prompt construction
        collector.step_start("prompt_build", task)
        prompt_text = f"Explain clearly: {task}"
        collector.step_end(prompt_text)

        # Step 2: LLM call
        collector.step_start("llm_generate", prompt_text)
        result = chain.invoke({"task": task})
        collector.step_end(result)

        # Step 3: post-processing
        collector.step_start("postprocess", result[:50])
        word_count = len(result.split())
        collector.step_end(f"{word_count} words")

        collector.end("success")
        print(f"  ✓ {task_id}: {task[:40]}... ({collector.traces[-1].total_duration:.1f}s)")
    except Exception as e:
        collector.step_end("", success=False, error=str(e))
        collector.end("failed")
        print(f"  ✗ {task_id}: {e}")

## Step 3 — Trace Analysis Dashboard

In [ ]:
# Build analysis dataframe
rows = []
for trace in collector.traces:
    for step in trace.steps:
        rows.append({
            "task_id": trace.task_id,
            "task": trace.task[:30],
            "step": step.name,
            "duration_s": step.duration_s,
            "tokens": step.tokens_est,
            "success": step.success,
        })

tdf = pd.DataFrame(rows)

print("TRACE ANALYSIS")
print("=" * 60)

# Per-task summary
task_summary = tdf.groupby("task_id").agg({
    "duration_s": "sum",
    "tokens": "sum",
    "success": "all",
}).round(3)
print("Per-task:")
print(task_summary.to_string())

# Per-step breakdown
print("\nPer-step averages:")
step_summary = tdf.groupby("step").agg({
    "duration_s": ["mean", "max"],
    "tokens": "mean",
}).round(3)
print(step_summary.to_string())

# Bottleneck identification
slowest = tdf.loc[tdf["duration_s"].idxmax()]
print(f"\nBottleneck: {slowest['step']} in {slowest['task_id']} ({slowest['duration_s']:.2f}s)")

## Step 4 — LLM-Powered Trace Diagnosis

In [ ]:
trace_json = json.dumps([{
    "task_id": t.task_id, "task": t.task, "status": t.status,
    "total_s": t.total_duration,
    "steps": [{"name": s.name, "duration": s.duration_s, "ok": s.success} for s in t.steps]
} for t in collector.traces], indent=2)

diagnosis_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyze these agent execution traces. Identify: "
     "1) Performance bottlenecks, 2) Failure patterns, "
     "3) Optimization opportunities, 4) Recommended changes."),
    ("human", "{traces}")
])
diagnosis_chain = diagnosis_prompt | llm | StrOutputParser()

diagnosis = diagnosis_chain.invoke({"traces": trace_json})
print("TRACE DIAGNOSIS")
print("=" * 50)
print(diagnosis[:500])

## What You Learned
- **Execution tracing** with step-level instrumentation
- **Performance profiling** per task and per step
- **Bottleneck identification** from trace data
- **LLM-powered diagnosis** of agent behavior